> 本 notebook 由 AI 翻译自英文原文，可能存在疏漏。以准确性为准时请参阅[英文原版](../../../../01-intro-to-ai-agents/code_samples/01-python-langchain-agent.ipynb)。代码单元与英文版完全相同，只翻译了说明文字。

# 第 01 课 - AI 智能体简介

欢迎来到 **AI Agents for Beginners** 课程的第一课！

**AI 智能体**（AI agent）是一种以大语言模型（LLM）作为推理引擎的程序，它能在真实世界中采取*行动*——调用 API、查询数据库或运行代码——来替用户完成目标。

在这个 notebook 里，你将构建自己的第一个智能体：一个推荐度假目的地的**旅行助手**。过程中你会学到：

1. 为任意 OpenAI 兼容接口构建聊天模型客户端。
2. 用 `@tool` 装饰器定义一个**工具**——它就是一个普通的 Python 函数。
3. 用 `create_agent` 创建智能体并运行它。
4. 逐 token 流式输出智能体的回复。

## 环境准备

前置条件：在仓库根目录运行 `pip install -r requirements.txt`，把 `.env.example` 复制为 `.env`，填入 `LLM_BASE_URL`、`LLM_API_KEY`、`LLM_MODEL`，然后运行 `python scripts/check_endpoint.py`。

下面的单元会从 `.env` 加载这些变量并构建聊天模型客户端。`ChatOpenAI` 使用 OpenAI Chat Completions 协议，所以同一段代码可以对接 DeepSeek、OpenAI、本地 Ollama 服务或任何其他兼容接口——只需要改这三个环境变量。`LLM_EXTRA_BODY` 用来传递可选的、特定于服务商的请求选项（对 DeepSeek 来说，它会关闭思考模式）。

In [1]:
import json
import os

from dotenv import find_dotenv, load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(find_dotenv())

missing = [name for name in ("LLM_BASE_URL", "LLM_API_KEY", "LLM_MODEL") if not os.environ.get(name)]
if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Copy .env.example to .env in the repository root and fill them in."
    )

llm = ChatOpenAI(
    model=os.environ["LLM_MODEL"],
    base_url=os.environ["LLM_BASE_URL"],
    api_key=os.environ["LLM_API_KEY"],
    extra_body=json.loads(os.environ.get("LLM_EXTRA_BODY") or "null"),
)
print(f"Model client ready: {os.environ['LLM_MODEL']} @ {os.environ['LLM_BASE_URL']}")

Model client ready: deepseek-v4-pro @ https://api.deepseek.com/v1


## 创建你的第一个智能体

一个智能体需要两样东西：

- **指令（Instructions）**，告诉它*自己是谁*以及*该如何行事*。在 LangChain 里，这些指令就是智能体的**系统提示词（system prompt）**。
- **工具（Tools）**——用 `@tool` 装饰的 Python 函数，智能体可以调用它们来获取信息或执行操作。函数的 **docstring 会成为工具描述**，模型在决定是否调用时会读取它；函数的类型注解则定义了参数。

下面我们定义一个简单的工具，返回一组热门度假目的地。当用户请求旅行推荐时，智能体就会使用这个工具。

In [2]:
from langchain.tools import tool


@tool
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona", "Paris", "Berlin", "Tokyo", "Sydney",
        "New York City", "Cairo", "Cape Town", "Rio de Janeiro", "Bali",
    ]

现在用 `create_agent` 把模型、工具和指令组装到一起。智能体运行的是一个**工具调用循环（tool-calling loop）**：把对话发给模型，执行模型请求的任何工具，把结果回传给模型，如此反复，直到模型以纯文本作答。

`agent.invoke` 返回完整的消息历史——用户消息、模型发起的工具调用、工具结果，以及最终回复。`reply_text` 从最后一条消息里取出文本（有些服务商返回的 content 是一组内容块的列表，而不是单个字符串）。

In [3]:
from langchain.agents import create_agent

agent = create_agent(
    llm,
    tools=[get_destinations],
    system_prompt=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)


def reply_text(result) -> str:
    """Return the text of the last message; content can be a string or a list of blocks."""
    content = result["messages"][-1].content
    if isinstance(content, list):
        return "".join(block.get("text", "") for block in content if isinstance(block, dict))
    return content


result = agent.invoke(
    {"messages": [{"role": "user", "content": "I'm looking for a warm beach destination. What do you recommend?"}]}
)
print(reply_text(result))

Based on your preference for a **warm beach destination**, here are my top recommendations from the available options:

1. **Bali** 🏝️ — The ultimate warm beach getaway. Expect tropical weather, stunning beaches (Kuta, Nusa Dua, Uluwatu), lush rice terraces, and a rich cultural scene. Perfect for both relaxation and adventure.

2. **Rio de Janeiro** 🌊 — Famous beaches like Copacabana and Ipanema, warm sunny weather year-round, vibrant culture, and incredible scenery (think Sugarloaf Mountain and Christ the Redeemer).

3. **Sydney** ☀️ — While it's on the temperate side, its famous beaches (Bondi, Manly) and warm summers make it a fantastic beach destination with plenty to do.

4. **Cape Town** 🌅 — Beautiful beaches like Camps Bay and Clifton, warm Mediterranean-style climate, and breathtaking scenery with Table Mountain as a backdrop.

5. **Barcelona** 🏖️ — Sunny Mediterranean beaches (Barceloneta) combined with amazing food, architecture, and nightlife.

**My top pick:** If you want a

## 流式输出回复

为了获得更好的交互体验，你可以**流式**接收智能体的回复。智能体不必等完整回复生成完毕，而是在生成过程中逐块产出文本。这在需要实时显示输出的聊天界面里尤其有用。

`agent.astream(..., stream_mode="messages")` 会为图产生的每一个消息块产出一对 `(token, metadata)`。我们只打印来自模型节点的块（工具调用和工具结果也会流经这条流）。Jupyter 支持顶层 `await`，所以下面的 `async for` 可以直接运行。

In [4]:
async for token, metadata in agent.astream(
    {"messages": [{"role": "user", "content": "Tell me about Tokyo as a travel destination"}]},
    stream_mode="messages",
):
    if metadata.get("langgraph_node") == "model" and getattr(token, "content", None):
        print(token.content, end="", flush=True)
print()

Tokyo is a truly incredible travel destination! Here's what makes it so special:

## 🏙️ Overview
Tokyo is a dazzling blend of ultra-modern technology and deep-rooted tradition. It's one of the world's most exciting cities, where you can find serene ancient temples just blocks away from neon-lit shopping streets and futuristic skyscrapers.

## ✨ Highlights

**Iconic Neighborhoods**
- **Shibuya** – Famous for the Shibuya Crossing, one of the world's busiest pedestrian intersections, plus amazing shopping and nightlife.
- **Shinjuku** – Entertainment district with vibrant nightlife, the quirky Golden Gai bars, and lovely Shinjuku Gyoen Garden.
- **Asakusa** – Home to the historic Sensō-ji Temple, Tokyo's oldest temple, and traditional Nakamise shopping street.
- **Harajuku** – The heart of youth culture and eclectic fashion, right next to the peaceful Meiji Shrine.

**Culture & History**
- Ancient temples and shrines alongside imperial gardens
- World-class museums like the Tokyo National

## 小结

在本课中你学会了：

- 用 `ChatOpenAI` **创建模型客户端**，它可以对接任何 OpenAI 兼容接口——服务商只是一个配置项。
- 用 `@tool` 装饰器**定义工具**，把一个普通的 Python 函数（连同它的 docstring）变成模型可以调用的东西。
- 用 `create_agent` **创建智能体**，把模型、工具和指令组装成一个工具调用循环。
- 用 `astream` **流式输出回复**，让 token 一到达就打印出来。

下一课我们会更深入地探索智能体框架，学习如何给智能体配备更强大的工具和多步推理能力。